# 🎨 PhotoEnhance — AI Photo Upscaler
**Безкоштовне покращення якості фото за допомогою Real-ESRGAN**

---

| Параметр | Значення |
|---|---|
| Модель | Real-ESRGAN |
| GPU | Google Colab T4 (16GB) |
| Масштаб | x2, x4, x8 |
| Формати | JPG, PNG, WEBP |
| Вартість | $0 |

> ⚠️ **Переконайся, що увімкнено GPU:** Runtime → Change runtime type → T4 GPU

## Як користуватись: 3 кроки
| Крок | Що робить |
|---|---|
| ▶ **Крок 1** | Встановлення, KeepAlive, перевірка GPU |
| ▶ **Крок 2** | Google Drive + завантаження фото |
| ▶ **Крок 3** | Обробка та скачування результату |


In [ ]:

# @title ⚙️ Крок 1: Встановлення, KeepAlive та перевірка GPU { display-mode: "form" }
# @markdown Запускай **один раз** на початку сесії.

import sys, os
import ipywidgets as widgets
from IPython.display import display, Javascript, HTML

# ── Прогрес-бар ──────────────────────────────────
_pb1 = widgets.IntProgress(
    value=0, min=0, max=100,
    description='⏳  0%',
    bar_style='info',
    style={'bar_color': '#4a90e2', 'description_width': '65px'},
    layout=widgets.Layout(width='100%', height='30px'),
)
_pb1_label = widgets.HTML('<span style="font-size:12px;color:#555;">🔎 Перевірка середовища...</span>')
display(widgets.VBox([_pb1, _pb1_label]))

# ════════════════════════════════════════════════
# 🔎 ПЕРЕВІРКИ ПЕРЕД КРОКОМ 1
# ════════════════════════════════════════════════
_ok = True

# 1) Python версія
_ver = sys.version_info
if _ver < (3, 8):
    print(f'❌ Python {_ver.major}.{_ver.minor} — потрібен 3.8+. Зміни runtime у Colab.')
    _ok = False
else:
    print(f'✅ Python {_ver.major}.{_ver.minor}.{_ver.micro}')

# 2) Середовище — Colab?
try:
    import google.colab
    print('✅ Середовище: Google Colab')
except ImportError:
    print('⚠️  Не Colab — деякі функції (Drive, files.download) можуть не працювати.')

# 3) Доступна RAM
try:
    with open('/proc/meminfo') as _m:
        _ram_kb = int([l for l in _m if 'MemAvailable' in l][0].split()[1])
    _ram_gb = _ram_kb / 1024**2
    if _ram_gb < 2:
        print(f'⚠️  Мало RAM: {_ram_gb:.1f} GB вільно — можливі помилки при обробці великих фото.')
    else:
        print(f'✅ RAM доступно: {_ram_gb:.1f} GB')
except Exception:
    print('ℹ️  RAM: не вдалося перевірити')

# 4) Місце на диску /content
try:
    import shutil as _sh
    _total, _used, _free = _sh.disk_usage('/content')
    _free_gb = _free / 1024**3
    if _free_gb < 3:
        print(f'⚠️  Мало місця на диску: {_free_gb:.1f} GB — може не вистачити для моделей (~1 GB) та результатів.')
    else:
        print(f'✅ Місце на диску: {_free_gb:.1f} GB вільно')
except Exception:
    print('ℹ️  Диск: не вдалося перевірити')

if not _ok:
    _pb1.bar_style = 'danger'
    _pb1.description = '❌  Помилка'
    raise SystemExit('❌ Усунь помилки вище перед запуском Кроку 1.')

_pb1.value = 5; _pb1.description = '🔎  5%'
_pb1_label.value = '<span style="font-size:12px;color:#555;">📦 Встановлення залежностей (basicsr, gfpgan, realesrgan)...</span>'
print('\n📦 Всі перевірки пройдено — починаємо встановлення...\n')

# ════════════════════════════════════════════════
# 1A. ВСТАНОВЛЕННЯ ЗАЛЕЖНОСТЕЙ + Real-ESRGAN
# ════════════════════════════════════════════════
print('📦 Встановлення залежностей...')

!pip install -q basicsr facexlib gfpgan
_pb1.value = 12; _pb1.description = '⬇️  12%'

!pip install -q realesrgan
_pb1.value = 18; _pb1.description = '⬇️  18%'

!pip install -q ipywidgets Pillow opencv-python-headless
_pb1.value = 22; _pb1.description = '⬇️  22%'
_pb1_label.value = '<span style="font-size:12px;color:#555;">🔧 Фікс torchvision сумісності...</span>'

# Фікс сумісності basicsr з torchvision >= 0.16
import sys
from types import ModuleType
import torchvision.transforms.functional as _tvf

_compat = ModuleType('torchvision.transforms.functional_tensor')
_compat.rgb_to_grayscale = _tvf.rgb_to_grayscale
sys.modules['torchvision.transforms.functional_tensor'] = _compat

_pb1.value = 25; _pb1.description = '📦  25%'
_pb1_label.value = '<span style="font-size:12px;color:#555;">📦 Клонування Real-ESRGAN...</span>'

import os
if not os.path.exists('/content/Real-ESRGAN'):
    !git clone -q https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN
    _pb1.value = 32; _pb1.description = '📦  32%'
    %cd /content/Real-ESRGAN
    !pip install -q -r requirements.txt
    _pb1.value = 40; _pb1.description = '📦  40%'
    !python setup.py develop -q
    _pb1.value = 45; _pb1.description = '📦  45%'
else:
    _pb1.value = 45; _pb1.description = '📦  45%'
    %cd /content/Real-ESRGAN

print('✅ Real-ESRGAN встановлено!\n')

# ════════════════════════════════════════════════
# 1B. CODEFORMER — для портретів і старих фото
# ════════════════════════════════════════════════
# CodeFormer (NeurIPS 2022): відновлення облич з контролем fidelity.
# fidelity=1.0 → вигляд близький до оригіналу (природна шкіра)
# fidelity=0.5 → більше "відновлення" (для сильно деградованих фото)
# GitHub: https://github.com/sczhou/CodeFormer

_pb1_label.value = '<span style="font-size:12px;color:#555;">🔬 Встановлення CodeFormer (NeurIPS 2022)...</span>'
print('🔬 Встановлення CodeFormer...')
if not os.path.exists('/content/CodeFormer'):
    _pb1.value = 48; _pb1.description = '🔬  48%'
    !git clone -q https://github.com/sczhou/CodeFormer.git /content/CodeFormer
    _pb1.value = 55; _pb1.description = '🔬  55%'
    %cd /content/CodeFormer
    !pip install -q -r requirements.txt
    _pb1.value = 62; _pb1.description = '🔬  62%'
    !python basicsr/setup.py develop -q
    _pb1.value = 68; _pb1.description = '🔬  68%'
    _pb1_label.value = '<span style="font-size:12px;color:#555;">⬇️ Завантаження ваг CodeFormer (facelib + weights)...</span>'
    # Завантажуємо pretrained моделі (facelib для детекції, CodeFormer для відновлення)
    !python scripts/download_pretrained_models.py facelib
    _pb1.value = 76; _pb1.description = '⬇️  76%'
    !python scripts/download_pretrained_models.py CodeFormer
    _pb1.value = 83; _pb1.description = '🔬  83%'
    print('✅ CodeFormer встановлено!\n')
else:
    _pb1.value = 83; _pb1.description = '🔬  83%'
    print('✅ CodeFormer вже є\n')

%cd /content/Real-ESRGAN

# ════════════════════════════════════════════════
# 1C. KEEPALIVE — захист від розриву з'єднання
# ════════════════════════════════════════════════
_pb1.value = 87; _pb1.description = '🔌  87%'
_pb1_label.value = '<span style="font-size:12px;color:#555;">🔌 Активація KeepAlive...</span>'

import threading

display(Javascript("""
(function startKeepAlive() {
    if (window._peKeepAlive) clearInterval(window._peKeepAlive);
    window._peKeepAlive = setInterval(function() {
        fetch('/api/kernels').catch(function(){});
    }, 45000);
    console.log('[PhotoEnhance] KeepAlive started — ping every 45s');
})();
"""))

_ka_stop = threading.Event()
def _heartbeat():
    c = 0
    while not _ka_stop.wait(timeout=50):
        c += 1
threading.Thread(target=_heartbeat, daemon=True, name='PE-KeepAlive').start()
print('🔌 KeepAlive активовано (45 сек браузер / 50 сек kernel)\n')

# ════════════════════════════════════════════════
# 1D. ПЕРЕВІРКА GPU
# ════════════════════════════════════════════════
_pb1.value = 92; _pb1.description = '🖥️  92%'
_pb1_label.value = '<span style="font-size:12px;color:#555;">🖥️ Перевірка GPU...</span>'

import subprocess
import torch

print('=' * 50)
print('🖥️  ІНФОРМАЦІЯ ПРО GPU')
print('=' * 50)

if torch.cuda.is_available():
    gpu_name   = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU знайдено: {gpu_name}')
    print(f'💾 VRAM: {gpu_memory:.1f} GB')
    print(f'🔧 CUDA версія: {torch.version.cuda}')
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,memory.free,temperature.gpu',
         '--format=csv,noheader,nounits'], capture_output=True, text=True)
    if result.returncode == 0:
        parts = result.stdout.strip().split(', ')
        if len(parts) >= 4:
            print(f'🌡️  Температура: {parts[3]}°C')
            print(f'📊 Вільна пам\'ять: {int(parts[2])//1024:.1f} GB / {int(parts[1])//1024:.1f} GB')
    print('\n✅ Все готово! Переходь до Кроку 2 ▶')
else:
    print('❌ GPU не знайдено!')
    print('👉 Runtime → Change runtime type → T4 GPU → Save → перезапусти Крок 1')

print('=' * 50)

# ════════════════════════════════════════════════
# ГОТОВО
# ════════════════════════════════════════════════
_pb1.value = 100; _pb1.description = '✅ 100%'
_pb1.bar_style = 'success'
_pb1_label.value = '<span style="font-size:12px;color:#2ecc71;font-weight:bold;">✅ Крок 1 завершено!</span>'

display(HTML("""
<div style="margin-top:12px;padding:14px 20px;background:linear-gradient(135deg,#1a1a2e,#16213e);
            border-radius:10px;border-left:5px solid #2ecc71;font-family:sans-serif;">
  <div style="color:#2ecc71;font-size:18px;font-weight:bold;">✅ Крок 1 завершено!</div>
  <div style="color:#ccc;margin-top:6px;font-size:14px;">
    📦 Real-ESRGAN &nbsp;✔&nbsp;&nbsp; 🔬 CodeFormer &nbsp;✔&nbsp;&nbsp; 🔌 KeepAlive &nbsp;✔&nbsp;&nbsp; 🖥️ GPU &nbsp;✔
  </div>
  <div style="margin-top:10px;font-size:15px;color:#4a90e2;font-weight:bold;">
    ↓ &nbsp;Гортай вниз і запускай <b style="color:#f9ca24;">Крок 2</b>
  </div>
</div>
"""))

# Автоматично переходимо до наступної комірки (Крок 2)
display(Javascript("try { google.colab.notebook.focusNextCell(); } catch(e) {}"))


In [ ]:

# @title 🖼️ Крок 2: Google Drive + Завантаження фото { display-mode: "form" }
# @markdown **Drive** — для автозбереження результатів. Можна пропустити — файл скачається напряму у Кроці 3.
# @markdown
# @markdown 💡 Потрібен особистий **@gmail.com**. 📱 На телефоні: якщо авторизація не йде — просто продовжуй.

import sys, os

# ════════════════════════════════════════════════
# 🔎 ПЕРЕВІРКИ ПЕРЕД КРОКОМ 2
# ════════════════════════════════════════════════
_ok = True

# 1) Крок 1 виконано? (torch має бути імпортовано)
if 'torch' not in sys.modules:
    print('❌ Крок 1 не виконано! Запусти Крок 1 і дочекайся "✅ Все готово".')
    _ok = False
else:
    print('✅ Крок 1 виконано')

# 2) GPU доступний?
if 'torch' in sys.modules:
    import torch as _t
    if not _t.cuda.is_available():
        print('⚠️  GPU не знайдено — обробка буде дуже повільною (10+ хв).')
        print('   👉 Runtime → Change runtime type → T4 GPU → Save → перезапусти з Кроку 1.')
    else:
        _vram = _t.cuda.get_device_properties(0).total_memory / 1024**3
        print(f'✅ GPU: {_t.cuda.get_device_name(0)} ({_vram:.1f} GB VRAM)')
        if _vram < 4:
            print('⚠️  VRAM < 4 GB — при x8 масштабі може виникнути OOM. Використовуй x2 або x4.')

# 3) Підтримувані бібліотеки встановлено?
for _lib in ['realesrgan', 'basicsr', 'cv2', 'PIL']:
    try:
        __import__(_lib)
        print(f'✅ {_lib} встановлено')
    except ImportError:
        print(f'❌ {_lib} не знайдено — перезапусти Крок 1.')
        _ok = False

# 4) CodeFormer встановлено?
if os.path.exists('/content/CodeFormer'):
    print('✅ CodeFormer встановлено')
else:
    print('⚠️  CodeFormer не знайдено — портретний режим буде недоступний (перезапусти Крок 1).')

# 5) Місце на диску
try:
    import shutil as _sh
    _, _, _free = _sh.disk_usage('/content')
    if _free / 1024**3 < 1:
        print('⚠️  Менше 1 GB на диску — може не вистачити для збереження результату.')
    else:
        print(f'✅ Диск: {_free/1024**3:.1f} GB вільно')
except Exception:
    pass

if not _ok:
    raise SystemExit('❌ Усунь помилки вище перед запуском Кроку 2.')

print('\n✅ Всі перевірки пройдено!\n')

from google.colab import drive
import os
import io
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from PIL import Image

# ════════════════════════════════════════════════
# ЛІМІТИ (особисте використання — максимальна якість)
# ════════════════════════════════════════════════
MAX_FILE_MB     = 100   # Максимальний розмір файлу в MB
PROCESS_TIMEOUT = 600   # Максимальний час обробки (10 хв)
# Примітка: авто-зменшення прибрано — обробляємо фото в оригінальному розмірі

TEMP_DIR = '/content/photoenhance_temp'
os.makedirs(TEMP_DIR, exist_ok=True)

# ════════════════════════════════════════════════
# 2A. GOOGLE DRIVE
# ════════════════════════════════════════════════
if os.path.isdir('/content/drive/MyDrive'):
    print('✅ Google Drive вже підключено.')
    OUTPUT_DIR = '/content/drive/MyDrive/PhotoEnhance_Results'
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    DRIVE_AVAILABLE = True
    print(f'📂 Збереження: {OUTPUT_DIR}\n')
else:
    try:
        print('🔗 Підключення Google Drive...')
        drive.mount('/content/drive')
        OUTPUT_DIR = '/content/drive/MyDrive/PhotoEnhance_Results'
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        DRIVE_AVAILABLE = True
        print(f'✅ Drive підключено: {OUTPUT_DIR}\n')
    except Exception as e:
        print(f'⚠️  Drive недоступний: {e}')
        print('   Файл скачається напряму у Кроці 3.\n')
        OUTPUT_DIR = TEMP_DIR
        DRIVE_AVAILABLE = False

# ════════════════════════════════════════════════
# 2B. АДАПТИВНІ СТИЛІ (мобільні)
# ════════════════════════════════════════════════
display(HTML("""
<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0">
<style>
  .pe-title { font-family: sans-serif; color: #1a1a2e; margin-bottom: 10px; }
  .pe-info  { font-family: monospace; background: #f0f4ff; padding: 10px;
               border-radius: 6px; border-left: 4px solid #4a90e2;
               word-break: break-word; font-size: 13px; }
  .pe-error { font-family: sans-serif; background: #fff0f0; padding: 10px;
               border-radius: 6px; border-left: 4px solid #e74c3c; font-size: 13px; }
  .pe-warn  { font-family: sans-serif; background: #fffbe6; padding: 10px;
               border-radius: 6px; border-left: 4px solid #f39c12; font-size: 13px; }
  .widget-button { min-height: 48px !important; font-size: 15px !important; }
  @media (max-width: 600px) {
    .widget-select select, .widget-dropdown select { width: 100% !important; font-size: 14px !important; }
    .widget-upload, .widget-button { width: 100% !important; min-height: 52px !important; font-size: 16px !important; }
    .widget-toggle-buttons .widget-toggle-button { font-size: 11px !important; padding: 4px 6px !important; }
    .widget-hbox { flex-wrap: wrap !important; gap: 6px !important; }
    .widget-vbox { width: 100% !important; max-width: 100% !important; }
  }
</style>
"""))

# ════════════════════════════════════════════════
# 2C. ІНТЕРФЕЙС ЗАВАНТАЖЕННЯ
# ════════════════════════════════════════════════
title_html = widgets.HTML(f"""
<div class="pe-title">
  <h2>🎨 PhotoEnhance MVP</h2>
  <p style="color:#666;">Безкоштовне AI покращення якості фото — powered by Real-ESRGAN + CodeFormer</p>
  <p style="color:#888;font-size:12px;">
    📏 Ліміт файлу: <b>{MAX_FILE_MB} MB</b> &nbsp;|&nbsp;
    ⏱️ Таймаут: <b>{PROCESS_TIMEOUT // 60} хв</b> &nbsp;|&nbsp;
    ⚙️ FP32 (max якість) &nbsp;|&nbsp;
    📐 Формати: JPG, PNG, WEBP
  </p>
</div>
""")

upload_btn = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.webp', multiple=False,
    description='📸 Вибрати фото',
    layout=widgets.Layout(width='auto', min_width='200px', min_height='48px')
)

# ── Тип фото — визначає алгоритм обробки ──
photo_type = widgets.ToggleButtons(
    options=[
        ('🧑 Портрет',             'portrait'),
        ('👟 Товар / Текстура',    'product'),
        ('🌄 Пейзаж',             'landscape'),
        ('📷 Старе фото',          'old_photo'),
        ('🎌 Аніме',               'anime'),
    ],
    value='product',
    description='Тип фото:',
    style={'button_width': 'auto', 'description_width': '80px'},
    layout=widgets.Layout(width='100%'),
)

photo_type_desc = widgets.HTML("""
<div class="pe-info" style="font-size:12px;">
  <b>🧑 Портрет</b> — CodeFormer: відновлює обличчя зберігаючи <i>справжню</i> шкіру (не воскову).<br>
  <b>👟 Товар / Текстура</b> — Real-ESRGAN: шкіра, тканина, гума — чітко і різко.<br>
  <b>🌄 Пейзаж</b> — Real-ESRGAN: деталі та краї без розмиття.<br>
  <b>📷 Старе фото</b> — CodeFormer (обличчя) + ESRGAN (фон): відновлення деградацій.<br>
  <b>🎌 Аніме</b> — спеціальна anime-модель: чіткі контури, чисті кольори.
</div>
""")

# ── Слайдер fidelity для портретів і старих фото ──
face_fidelity = widgets.FloatSlider(
    value=0.8, min=0.0, max=1.0, step=0.05,
    description='Fidelity:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='100%', max_width='500px'),
    readout_format='.2f',
)
fidelity_desc = widgets.HTML("""
<div style="font-size:11px;color:#666;margin-top:-8px;">
  <b>0.0</b> = максимальне відновлення (для дуже деградованих) &nbsp;|&nbsp;
  <b>0.8</b> = природна шкіра (рекомендовано) &nbsp;|&nbsp;
  <b>1.0</b> = майже без змін обличчя
</div>
""")

fidelity_box = widgets.VBox([face_fidelity, fidelity_desc])
fidelity_box.layout.display = 'none'  # приховано за замовчуванням

def on_photo_type_change(change):
    if change['new'] in ('portrait', 'old_photo'):
        fidelity_box.layout.display = ''
    else:
        fidelity_box.layout.display = 'none'

photo_type.observe(on_photo_type_change, names='value')

scale_options = widgets.ToggleButtons(
    options=[('× 2  (швидше)', 2), ('× 4  (оптимально)', 4), ('× 8  (max розмір)', 8)],
    value=4, description='Масштаб:',
    style={'button_width': 'auto'}
)

model_options = widgets.Dropdown(
    options=[
        ('🖼️  Загальні фото (реальні сцени)', 'RealESRGAN_x4plus'),
        ('🎌 Аніме / Ілюстрації', 'RealESRGAN_x4plus_anime_6B'),
        ('📷 Старі / деградовані фото', 'ESRGAN_SRx4_DF2KOST'),
        ('⚡ Швидкий режим (x2)', 'RealESRGAN_x2plus'),
    ],
    value='RealESRGAN_x4plus', description='Модель:',
    layout=widgets.Layout(width='100%', max_width='460px')
)

model_note = widgets.HTML("""
<div style="font-size:11px;color:#888;">
  ⚠️ Модель ігнорується для режимів «Портрет», «Старе фото» і «Аніме» — там використовується оптимальна автоматично.
</div>
""")

preview_output = widgets.Output()
info_label = widgets.HTML('<div class="pe-info">📁 Фото не вибрано</div>')

def on_upload_change(change):
    if upload_btn.value:
        for fname, fdata in upload_btn.value.items():
            size_kb = len(fdata['content']) / 1024
            size_mb = size_kb / 1024

            # ── Перевірка розміру файлу ──
            if size_mb > MAX_FILE_MB:
                info_label.value = (
                    f'<div class="pe-error">'
                    f'❌ <b>Файл занадто великий: {size_mb:.1f} MB</b><br>'
                    f'Максимальний розмір: <b>{MAX_FILE_MB} MB</b>.<br>'
                    f'Стисни фото або обери інше.'
                    f'</div>'
                )
                with preview_output:
                    clear_output()
                return

            # ── Перевірка, що це зображення ──
            try:
                img = Image.open(io.BytesIO(fdata['content']))
            except Exception:
                info_label.value = '<div class="pe-error">❌ Файл не є зображенням — обери JPG/PNG/WEBP.</div>'
                with preview_output:
                    clear_output()
                return

            w, h = img.size
            total_px = w * h
            mpx = total_px / 1_000_000

            info_label.value = (
                f'<div class="pe-info">'
                f'✅ <b>{fname}</b><br>'
                f'📐 {w}×{h} px &nbsp;|&nbsp; {size_kb:.0f} KB &nbsp;|&nbsp; {img.mode} &nbsp;|&nbsp; {mpx:.1f} Mpx'
                f'</div>'
            )

            with preview_output:
                clear_output(wait=True)
                img_thumb = img.copy()
                img_thumb.thumbnail((280, 280))
                import matplotlib.pyplot as plt
                fig, ax = plt.subplots(figsize=(3.5, 3.5))
                ax.imshow(img_thumb)
                ax.set_title('Оригінал (прев\'ю)', fontsize=10)
                ax.axis('off')
                plt.tight_layout()
                plt.show()

upload_btn.observe(on_upload_change, names='value')

ui = widgets.VBox([
    title_html,
    upload_btn, info_label, preview_output,
    widgets.HTML('<hr>'),
    photo_type, photo_type_desc, fidelity_box,
    widgets.HTML('<hr>'),
    model_options, model_note, scale_options,
    widgets.HTML('<hr style="margin-bottom:6px">'),
    widgets.HTML('<b>✅ Готово? Запускай Крок 3 ▶</b>'),
], layout=widgets.Layout(width='100%', max_width='650px'))

display(ui)


In [ ]:

# @title ⚡ Крок 3: Обробка фото + Результат + Скачування { display-mode: "form" }
# @markdown Запускай після вибору фото у Кроці 2.

# ════════════════════════════════════════════════════════════════════
# ТЕХНІЧНЕ ЗАВДАННЯ (ТЗ) — Стратегія покращення залежно від типу фото
# ════════════════════════════════════════════════════════════════════
#
# 🧑 ПОРТРЕТ (portrait)
#    Мета: природна шкіра з порами — НЕ воскова/plastic.
#    Рішення: CodeFormer (fidelity=0.8) → природна шкіра.
#    + face_upsample через Real-ESRGAN → обличчя ×outscale, не розмите.
#    + ESRGAN фон tile=512, FP32, tile_pad=32.
#
# 👟 ТОВАР / ТЕКСТУРА (product)
#    Мета: матеріал чіткий, без швів між тайлами.
#    Рішення: Real-ESRGAN x4plus, tile=512, tile_pad=32, FP32.
#
# 🌄 ПЕЙЗАЖ / АРХІТЕКТУРА (landscape)
#    Рішення: Real-ESRGAN x4plus, tile=512, tile_pad=32, FP32.
#
# 📷 СТАРЕ / ДЕГРАДОВАНЕ ФОТО (old_photo)
#    Рішення: CodeFormer fidelity=0.5 (більше відновлення) + ESRGAN фон.
#
# 🎌 АНІМЕ / ІЛЮСТРАЦІЯ (anime)
#    Рішення: RealESRGAN_x4plus_anime_6B — навчена на аніме.
#
# 🔑 КЛЮЧОВІ ПАРАМЕТРИ:
#    - tile=512, tile_pad=32 → менше швів між тайлами
#    - half=False (FP32) → точніші розрахунки
#    - face_upsample → обличчя та фон однакової роздільності
#    - БЕЗ авто-зменшення вхідного фото
# ════════════════════════════════════════════════════════════════════

import os, io, time, urllib.request, sys, shutil, threading, base64, re
from types import ModuleType

# Фікс torchvision сумісності
if 'torchvision.transforms.functional_tensor' not in sys.modules:
    import torchvision.transforms.functional as _tvf
    _compat = ModuleType('torchvision.transforms.functional_tensor')
    _compat.rgb_to_grayscale = _tvf.rgb_to_grayscale
    sys.modules['torchvision.transforms.functional_tensor'] = _compat

import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from PIL import Image
from IPython.display import display, HTML, clear_output
from torchvision.transforms.functional import normalize as tv_normalize

# Фікс PIL DecompressionBombError для великих зображень
Image.MAX_IMAGE_PIXELS = None

# Шляхи до проєктів
for _p in ['/content/Real-ESRGAN', '/content/CodeFormer']:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ════════════════════════════════════════════════
# ПАРАМЕТРИ (особисте використання — max якість)
# ════════════════════════════════════════════════
_max_file_mb   = MAX_FILE_MB       if 'MAX_FILE_MB'      in globals() else 100
_proc_timeout  = PROCESS_TIMEOUT   if 'PROCESS_TIMEOUT'  in globals() else 600
_photo_type    = photo_type.value  if 'photo_type'       in globals() else 'product'
_face_fidelity = face_fidelity.value if 'face_fidelity'  in globals() else 0.8

# ════════════════════════════════════════════════
# 3A. КОНФІГУРАЦІЯ МОДЕЛЕЙ
# ════════════════════════════════════════════════
MODEL_URLS = {
    'RealESRGAN_x4plus': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth', 4),
    'RealESRGAN_x4plus_anime_6B': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth', 4),
    'RealESRGAN_x2plus': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth', 2),
    'ESRGAN_SRx4_DF2KOST': (
        'https://github.com/xinntao/ESRGAN/releases/download/v0.1.1/ESRGAN_SRx4_DF2KOST_official-ff704c30.pth', 4),
}

MODELS_DIR = '/content/models'
os.makedirs(MODELS_DIR, exist_ok=True)

def download_model(model_name):
    url, _ = MODEL_URLS[model_name]
    model_path = os.path.join(MODELS_DIR, f'{model_name}.pth')
    if not os.path.exists(model_path):
        print(f'⬇️  Завантаження моделі {model_name}...')
        urllib.request.urlretrieve(url, model_path)
        print(f'✅ Модель завантажено')
    else:
        print(f'✅ Модель вже є: {model_name}')
    return model_path

def build_upsampler(model_name, outscale):
    """Real-ESRGAN з tile=512, tile_pad=32 і FP32 — максимальна якість текстур."""
    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet
    model_path = download_model(model_name)
    _, model_scale = MODEL_URLS[model_name]
    num_block = 6 if 'anime_6B' in model_name else 23
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
                    num_block=num_block, num_grow_ch=32, scale=model_scale)
    return RealESRGANer(
        scale=model_scale, model_path=model_path, model=model,
        tile=512,        # більший tile → більше контексту → менше швів
        tile_pad=32,     # більший pad → менше шовних артефактів на границях тайлів
        pre_pad=0,
        half=False,      # FP32 — точніші розрахунки
    )

# ════════════════════════════════════════════════
# 3B. ВИБІР МОДЕЛІ ЗАЛЕЖНО ВІД ТИПУ ФОТО
# ════════════════════════════════════════════════
def resolve_model(photo_type, selected_model, selected_scale):
    """Повертає оптимальні (model_name, scale) для типу фото."""
    if photo_type == 'anime':
        return 'RealESRGAN_x4plus_anime_6B', min(selected_scale, 4)
    elif photo_type in ('portrait', 'old_photo'):
        return 'RealESRGAN_x4plus', selected_scale  # для фону в CodeFormer pipeline
    else:
        return selected_model, selected_scale

# ════════════════════════════════════════════════
# 3C. CODEFORMER PIPELINE — ПОРТРЕТ / СТАРЕ ФОТО
# ════════════════════════════════════════════════
def build_codeformer_net():
    """
    Завантажує CodeFormer (NeurIPS 2022).
    КРИТИЧНО: спочатку імпортуємо codeformer_arch, щоб він зареєструвався
    в ARCH_REGISTRY через @ARCH_REGISTRY.register декоратор.
    """
    import importlib as _il
    _il.import_module('basicsr.archs.codeformer_arch')
    from basicsr.utils.registry import ARCH_REGISTRY
    from basicsr.utils.download_util import load_file_from_url
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    net = ARCH_REGISTRY.get('CodeFormer')(
        dim_embd=512, codebook_size=1024, n_head=8, n_layers=9,
        connect_list=['32', '64', '128', '256']
    ).to(device)
    ckpt_path = load_file_from_url(
        url='https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth',
        model_dir='/content/CodeFormer/weights/CodeFormer',
        progress=True, file_name=None
    )
    ckpt = torch.load(ckpt_path)['params_ema']
    net.load_state_dict(ckpt)
    net.eval()
    return net, device

def enhance_portrait(input_path, output_path, outscale, fidelity_weight, _pb):
    """
    CodeFormer pipeline:
    1. RetinaFace детектує обличчя
    2. CodeFormer відновлює кожне (fidelity → натуральна шкіра)
    3. face_upsample: відновлені обличчя апскейляться через Real-ESRGAN
       (без цього — фон ×outscale, обличчя 512px → розмите після склеювання)
    4. Real-ESRGAN апскейлить фон (tile=512, tile_pad=32, FP32)
    5. Відновлені обличчя вклеюються в апскейлений фон
    """
    from basicsr.utils import img2tensor, tensor2img
    from facelib.utils.face_restoration_helper import FaceRestoreHelper

    img = cv2.imread(input_path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'Не вдалося прочитати: {input_path}')
    h, w = img.shape[:2]
    print(f'📐 Вхід: {w}×{h} px  |  CodeFormer  |  fidelity={fidelity_weight}  |  ×{outscale}')

    _pb.value = 5;  _pb.description = '⬇️   5%'

    # Завантажуємо CodeFormer
    net, device = build_codeformer_net()
    _pb.value = 15; _pb.description = '🔬  15%'

    # Завантажуємо Real-ESRGAN для фону + face_upsample (tile=512, tile_pad=32, FP32)
    bg_upsampler = build_upsampler('RealESRGAN_x4plus', outscale)
    _pb.value = 25; _pb.description = '⬇️  25%'

    # FaceRestoreHelper: детекція, вирівнювання, вклеювання
    face_helper = FaceRestoreHelper(
        outscale, face_size=512, crop_ratio=(1, 1),
        det_model='retinaface_resnet50',
        save_ext='png', use_parse=True, device=device
    )
    face_helper.clean_all()  # очищаємо стан між запусками
    face_helper.read_image(img)
    num_faces = face_helper.get_face_landmarks_5(
        only_center_face=False, resize=640, eye_dist_threshold=5
    )
    print(f'🧑 Знайдено облич: {num_faces}')

    if num_faces == 0:
        print('⚠️  Обличчя не знайдено — застосовую Real-ESRGAN без CodeFormer.')
        _pb.value = 30
        return enhance_standard(input_path, output_path, 'RealESRGAN_x4plus', outscale, _pb)

    face_helper.align_warp_face()
    _pb.value = 35; _pb.description = '🔬  35%'

    # ── Відновлення кожного обличчя через CodeFormer ──
    n_faces = len(face_helper.cropped_faces)
    for idx, cropped_face in enumerate(face_helper.cropped_faces):
        cropped_face_t = img2tensor(cropped_face / 255., bgr2rgb=True, float32=True)
        tv_normalize(cropped_face_t, (0.5, 0.5, 0.5), (0.5, 0.5, 0.5), inplace=True)
        cropped_face_t = cropped_face_t.unsqueeze(0).to(device)
        try:
            with torch.no_grad():
                output = net(cropped_face_t, w=fidelity_weight, adain=True)[0]
                restored_face = tensor2img(output, rgb2bgr=True, min_max=(-1, 1))
            del output
            torch.cuda.empty_cache()
        except Exception as err:
            print(f'⚠️  Обличчя {idx+1} — помилка: {err}, використовую оригінал.')
            restored_face = tensor2img(cropped_face_t.squeeze(0), rgb2bgr=True, min_max=(-1, 1))
        restored_face = restored_face.astype('uint8')
        face_helper.add_restored_face(restored_face)
        pct = 35 + int((idx + 1) / n_faces * 35)
        _pb.value = pct; _pb.description = f'🔬 {pct:>3}%'

    # ── face_upsample: апскейл відновлених облич через Real-ESRGAN ──
    # Без цього CodeFormer дає 512×512, а фон ×outscale → обличчя буде розмитим після paste
    _pb.value = 72; _pb.description = '🔬  72%'
    face_helper.face_upsampler = bg_upsampler  # той самий upsampler що й для фону

    # ── Апскейл фону Real-ESRGAN (tile=512, tile_pad=32, FP32) ──
    _pb.value = 78; _pb.description = '🌄  78%'
    bg_img = bg_upsampler.enhance(img, outscale=outscale)[0]

    _pb.value = 90; _pb.description = '🔧  90%'

    # ── Склеювання облич з фоном ──
    try:
        face_helper.get_inverse_affine(None)
    except TypeError:
        # Новий API деяких версій FaceRestoreHelper
        face_helper.get_inverse_affine(save_inverse_affine_input_size=False)

    restored_img = face_helper.paste_faces_to_input_image(
        upsample_img=bg_img, draw_box=False
    )
    cv2.imwrite(output_path, restored_img)
    h_out, w_out = restored_img.shape[:2]
    return {'orig_size': (w, h), 'out_size': (w_out, h_out), 'output_path': output_path}

# ════════════════════════════════════════════════
# 3D. СТАНДАРТНИЙ Real-ESRGAN PIPELINE
# ════════════════════════════════════════════════
def enhance_standard(input_path, output_path, model_name, outscale, _pb):
    """
    Real-ESRGAN з tile=512, tile_pad=32 і FP32 — максимальна текстурна деталізація.
    Без жодного авто-зменшення. Без face processing.
    """
    img = cv2.imread(input_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f'Не вдалося прочитати: {input_path}')
    h, w = img.shape[:2]
    print(f'📐 Вхід: {w}×{h} px  |  {model_name}  |  ×{outscale}  |  tile=512  tile_pad=32  FP32')

    _pb.value = 10; _pb.description = '⬇️  10%'
    upsampler = build_upsampler(model_name, outscale)

    _real_stdout = sys.stdout

    class _TileCapture:
        """Перехоплює рядки "Tile X/Y", оновлює прогрес. Решту пропускає."""
        def write(self, text):
            m = re.search(r'Tile\s+(\d+)/(\d+)', text)
            if m:
                cur, tot = int(m.group(1)), int(m.group(2))
                pct = 15 + int(cur / tot * 80)
                _pb.value = min(pct, 95)
                _pb.description = f'🔄 {min(pct, 95):>3}%'
            else:
                _real_stdout.write(text)
        def flush(self): _real_stdout.flush()
        def __getattr__(self, n): return getattr(_real_stdout, n)

    sys.stdout = _TileCapture()
    _result = [None]
    _error  = [None]

    def _run():
        try:
            out, _ = upsampler.enhance(img, outscale=outscale)
            _result[0] = out
        except Exception as e:
            _error[0] = e

    _thread = threading.Thread(target=_run, daemon=True)
    _thread.start()
    _thread.join(timeout=_proc_timeout)
    sys.stdout = _real_stdout

    if _thread.is_alive():
        raise TimeoutError(f'⏱️  Обробка перевищила {_proc_timeout // 60} хв!')
    if _error[0]:
        raise _error[0]

    output_img = _result[0]
    h_out, w_out = output_img.shape[:2]
    cv2.imwrite(output_path, output_img)
    return {'orig_size': (w, h), 'out_size': (w_out, h_out), 'output_path': output_path}

# ════════════════════════════════════════════════
# 🔎 ПЕРЕВІРКИ ПЕРЕД КРОКОМ 3
# ════════════════════════════════════════════════
_ok = True

# 1) Кроки 1 і 2 виконано?
for _var, _name in [('upload_btn', 'Крок 2'), ('OUTPUT_DIR', 'Крок 2')]:
    if _var not in globals():
        print(f'❌ {_name} не виконано! Запусти кроки по порядку.')
        _ok = False

if 'torch' not in sys.modules:
    print('❌ Крок 1 не виконано!')
    _ok = False

if _ok:
    print('✅ Кроки 1 і 2 виконано')

# 2) Фото завантажено?
if _ok:
    if not upload_btn.value:
        print('❌ Фото не вибрано! Поверніться до Кроку 2 і натисніть "📸 Вибрати фото".')
        _ok = False
    else:
        _fname = list(upload_btn.value.keys())[0]
        _fdata = list(upload_btn.value.values())[0]
        _size_mb = len(_fdata['content']) / 1024**2
        print(f'✅ Фото: {_fname} ({_size_mb:.1f} MB)')

        if _size_mb > _max_file_mb:
            print(f'❌ Файл {_size_mb:.1f} MB > ліміту {_max_file_mb} MB.')
            _ok = False
        elif _size_mb == 0:
            print('❌ Файл порожній.')
            _ok = False

        if _ok:
            try:
                _img_check = Image.open(io.BytesIO(_fdata['content']))
                _w, _h = _img_check.size
                print(f'✅ Розмір: {_w}×{_h} px ({_w*_h/1_000_000:.1f} Mpx) | режим: {_img_check.mode}')
                if _w < 50 or _h < 50:
                    print('⚠️  Зображення < 50px — результат може бути поганим.')
            except Exception as _e:
                print(f'❌ Не вдалося відкрити зображення: {_e}')
                _ok = False

# 3) GPU пам'ять
if _ok and 'torch' in sys.modules:
    import torch as _t
    if _t.cuda.is_available():
        _t.cuda.empty_cache()
        _free_vram = (_t.cuda.get_device_properties(0).total_memory
                      - _t.cuda.memory_allocated()) / 1024**3
        print(f'✅ Вільна VRAM: {_free_vram:.1f} GB')
        if _free_vram < 1.5:
            print('⚠️  Мало VRAM — Runtime → Restart runtime → повтори з Кроку 1.')
    else:
        print('⚠️  GPU недоступний — обробка на CPU (дуже повільно).')

# 4) Місце на диску
if _ok:
    try:
        _, _, _free_d = shutil.disk_usage('/content')
        if _free_d / 1024**3 < 0.5:
            print('❌ Менше 500 MB на диску.')
            _ok = False
        else:
            print(f'✅ Диск: {_free_d/1024**3:.1f} GB вільно')
    except Exception:
        pass

# 5) CodeFormer перевірка для портретних режимів
if _ok and _photo_type in ('portrait', 'old_photo'):
    if not os.path.exists('/content/CodeFormer'):
        print('❌ CodeFormer не встановлено — перезапусти Крок 1.')
        _ok = False
    else:
        print('✅ CodeFormer готовий')

if not _ok:
    raise SystemExit('❌ Усунь помилки вище перед запуском Кроку 3.')

_type_labels = {
    'portrait':  '🧑 Портрет',
    'product':   '👟 Товар / Текстура',
    'landscape': '🌄 Пейзаж / Архітектура',
    'old_photo': '📷 Старе фото',
    'anime':     '🎌 Аніме',
}

print(f'\n✅ Тип фото: {_type_labels.get(_photo_type, _photo_type)}')
if _photo_type in ('portrait', 'old_photo'):
    _fidelity_hint = 'природна шкіра' if _face_fidelity >= 0.8 else 'більше відновлення'
    print(f'   CodeFormer fidelity: {_face_fidelity} ({_fidelity_hint})')
print(f'   Масштаб: ×{scale_options.value}  |  Таймаут: {_proc_timeout//60} хв  |  FP32  tile=512  tile_pad=32\n')

# ════════════════════════════════════════════════
# 3E. ЗАПУСК ОБРОБКИ
# ════════════════════════════════════════════════
from google.colab import files

selected_model = model_options.value
selected_scale = scale_options.value
final_model, final_scale = resolve_model(_photo_type, selected_model, selected_scale)

TEMP_DIR = '/content/photoenhance_temp'
os.makedirs(TEMP_DIR, exist_ok=True)

for fname, fdata in upload_btn.value.items():
    input_path = f'{TEMP_DIR}/{fname}'
    with open(input_path, 'wb') as f:
        f.write(fdata['content'])

    base, _ = os.path.splitext(fname)
    result_name  = f'{base}_enhanced_x{final_scale}.png'
    local_output = f'{TEMP_DIR}/{result_name}'
    drive_output = f'{OUTPUT_DIR}/{result_name}'

    print('=' * 60)
    print(f'🎨 PhotoEnhance — {fname}')
    if _photo_type in ('portrait', 'old_photo'):
        print(f'   CodeFormer (fidelity={_face_fidelity}) + face_upsample + ESRGAN фон  |  ×{final_scale}')
    else:
        print(f'   {final_model}  |  ×{final_scale}  |  FP32  tile=512  tile_pad=32')
    print('=' * 60)

    # ── Прогрес-бар ──
    _pb = widgets.IntProgress(
        value=0, min=0, max=100,
        description='⏳  0%',
        bar_style='info',
        style={'bar_color': '#4a90e2', 'description_width': '65px'},
        layout=widgets.Layout(width='100%', height='30px'),
    )
    display(_pb)

    t = time.time()
    try:
        if _photo_type in ('portrait', 'old_photo'):
            stats = enhance_portrait(
                input_path, local_output,
                final_scale, _face_fidelity, _pb
            )
        else:
            stats = enhance_standard(
                input_path, local_output,
                final_model, final_scale, _pb
            )
        _pb.value = 100
        _pb.description = '✅ 100%'
        _pb.bar_style = 'success'

    except TimeoutError as _te:
        _pb.bar_style = 'danger'
        _pb.description = '⏱️ Timeout'
        print(str(_te))
        raise SystemExit('❌ Таймаут. Спробуй: менший масштаб (x2) або менший файл.')
    except Exception as _ex:
        _pb.bar_style = 'danger'
        _pb.description = '❌ Помилка'
        raise

    elapsed = time.time() - t
    print(f'✅ Готово за {elapsed:.1f} сек  →  {stats["out_size"][0]}×{stats["out_size"][1]} px')

    if DRIVE_AVAILABLE and drive_output != local_output:
        shutil.copy2(local_output, drive_output)
        print(f'☁️  Збережено на Drive: {drive_output}')
    else:
        print('💾 Drive не підключено — скачується нижче')

# ════════════════════════════════════════════════
# 3F. BEFORE / AFTER + СТАТИСТИКА
# ════════════════════════════════════════════════
orig_img = Image.open(input_path).convert('RGB')
enh_img  = Image.open(local_output).convert('RGB')

# Зменшуємо лише для відображення — файл залишається оригінальним
_MAX_DISP = 2000
if max(enh_img.size) > _MAX_DISP:
    enh_img.thumbnail((_MAX_DISP, _MAX_DISP), Image.LANCZOS)
if max(orig_img.size) > _MAX_DISP:
    orig_img.thumbnail((_MAX_DISP, _MAX_DISP), Image.LANCZOS)

fig = plt.figure(figsize=(10, 5), facecolor='#1a1a2e')
gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.03)

ax1 = fig.add_subplot(gs[0])
ax1.imshow(orig_img)
ax1.set_title(f'ОРИГІНАЛ\n{stats["orig_size"][0]}×{stats["orig_size"][1]} px',
              color='white', fontsize=10, pad=8)
ax1.axis('off')
ax1.text(0.02, 0.02, '◀ BEFORE', transform=ax1.transAxes,
         color='white', fontsize=9, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#e74c3c', alpha=0.85))

ax2 = fig.add_subplot(gs[1])
ax2.imshow(enh_img)
ax2.set_title(f'ПОКРАЩЕНО (×{final_scale})\n{stats["out_size"][0]}×{stats["out_size"][1]} px',
              color='white', fontsize=10, pad=8)
ax2.axis('off')
ax2.text(0.02, 0.02, 'AFTER ▶', transform=ax2.transAxes,
         color='white', fontsize=9, fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='#2ecc71', alpha=0.85))

_type_label = _type_labels.get(_photo_type, _photo_type)
fig.suptitle(f'🎨 PhotoEnhance — {_type_label}  |  {elapsed:.1f}s  |  FP32  tile=512  tile_pad=32',
             color='white', fontsize=11, fontweight='bold', y=1.01)

comparison_path = f'{TEMP_DIR}/comparison.png'
plt.savefig(comparison_path, dpi=100, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

pixel_increase = (stats['out_size'][0] * stats['out_size'][1]) / \
                 (stats['orig_size'][0] * stats['orig_size'][1])

print('\n' + '=' * 60)
print('📊 СТАТИСТИКА')
print('=' * 60)
print(f'  📥 Оригінал:  {stats["orig_size"][0]:>5} × {stats["orig_size"][1]} px')
print(f'  📤 Результат: {stats["out_size"][0]:>5} × {stats["out_size"][1]} px')
print(f'  🔍 Масштаб:   ×{final_scale}  |  📈 Пікселів: ×{pixel_increase:.1f}')
print(f'  ⚙️  Режим:     FP32  |  tile=512  tile_pad=32  |  ⏱️  {elapsed:.1f} сек')
if _photo_type in ('portrait', 'old_photo'):
    print(f'  🔬 CodeFormer fidelity: {_face_fidelity}  |  face_upsample ✅  |  🤖 {final_model} (фон)')
else:
    print(f'  🤖 Модель:    {final_model}')
print('=' * 60)

# ════════════════════════════════════════════════
# 3G. СКАЧУВАННЯ (mobile-friendly base64 links)
# ════════════════════════════════════════════════
def make_download_link(filepath, label, mime='image/png'):
    with open(filepath, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    filename = os.path.basename(filepath)
    return (
        f'<a href="data:{mime};base64,{b64}" download="{filename}" '
        f'style="display:inline-block;margin:8px 4px;padding:12px 20px;'
        f'background:#2ecc71;color:#fff;font-weight:bold;font-size:15px;'
        f'border-radius:8px;text-decoration:none;min-width:200px;text-align:center;">'
        f'⬇️ {label}</a>'
    )

display(HTML("""
<div style="background:#f0f4ff;border-radius:8px;padding:12px;margin-top:10px;
            border-left:4px solid #4a90e2;font-family:sans-serif;">
  <b>📱 На телефоні:</b> натисни кнопку нижче → браузер збереже файл.<br>
  <small>Якщо не спрацює — натисни й утримуй зображення вище → "Зберегти зображення".</small>
</div>
""" + make_download_link(local_output, 'Зберегти покращене фото')
   + make_download_link(comparison_path, 'Зберегти порівняння')))

# Резервний варіант через google.colab (десктоп)
try:
    files.download(local_output)
    files.download(comparison_path)
except Exception:
    pass

print('\n✅ Готово!')
